# Hammoud/Ulrich atlas mapping — human arm, mouse arm, and gene drivers, in one verifiable notebook

**What this is.** All of the Ulrich et al. 2024 (Hammoud lab, PNAS 121:e2404775121) atlas-mapping
work for the paper revision, compiled so a reader can go through it top to bottom and check every
claim. It covers (§2) the human arm, (§3) the mouse/XMens arm, (§4) the gene-driver decomposition,
and (§5) the cross-arm synthesis. Generated by `fib_mapping/make_notebook_combined.py`.

**The two reviewer comments this addresses:**

1. *Do our data recapitulate biology the reviewer already knows (C7+ fibroblasts etc.)?*
2. *"What do you mean you only sequenced 2 humans?"*

**The atlas.** Ulrich 2024: full-thickness uterus, single-**cell**, 20 donors, 39 subtypes.
**Our data.** Single-**nucleus** + Slide-tags spatial. This asymmetry is why the headline metric is
the double-centred ("contrastive") cosine rather than raw correlation: raw correlation carries a
shared dataset-level offset across the nucleus/cell boundary (and, for the mouse arm, a species
offset on top of it).

**The one rule.** This project's cross-species work accumulated ~15 corrections, every one of which
moved toward a tidier story than the data supported. So every headline number in the prose below has
a `verify()` cell that recomputes it from the source CSV/JSON and prints PASS/MISMATCH; the notebook
ends with `check_summary()`. Prose drift found *during this compilation* is reported in §2.7 and §6
rather than silently corrected. Nothing in `fib_centroid_results/` or `fib_centroid_results_mouse/`
is recomputed here — those runs are finished; this notebook reads, verifies, and presents them.

## 0. Setup, self-check helper, and the input manifest

In [ ]:
import json, os, re, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

HAM  = Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/hammoud")
OUT  = HAM / "fib_centroid_results"          # human arm, finished 2026-08-30
MOUT = HAM / "fib_centroid_results_mouse"    # mouse arm,  finished 2026-08-30
H5MM = Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/deliverables/"
            "2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad")
sys.path.insert(0, str(HAM / "fib_mapping"))
from hammoud_fib_mapping import FIB_SUBTYPES, SHORT   # constants only; no computation

# ---- the house self-check helper, copied verbatim from notebooks/verify/ ----
_checks = []
def verify(label, got, expected, tol=0.02, note=""):
    """Recompute vs recorded. Prints PASS/MISMATCH and accumulates a summary."""
    try:
        ok = (abs(float(got) - float(expected)) <= tol * max(1.0, abs(float(expected)))
              if expected is not None else None)
    except (TypeError, ValueError):
        ok = (got == expected)
    tag = "PASS " if ok else ("MISMATCH" if ok is False else "n/a  ")
    _checks.append((label, got, expected, tag))
    print(f"[{tag}] {label}: recomputed={got}  recorded={expected}" + (f"   ({note})" if note else ""))
    return ok

def check_summary():
    d = pd.DataFrame(_checks, columns=["claim","recomputed","recorded","status"])
    n_bad = int((d.status.str.strip()=="MISMATCH").sum())
    print(f"\n{len(d)} checks | {n_bad} MISMATCH")
    return d

In [ ]:
# Every input this notebook consumes, with mtime and size, printed by the notebook itself.
INPUTS = {}
for name in [
    "resolution_sweep.csv", "resolution_stability.csv", "cluster_sizes.csv",
    "similarity_fib_contrastive_cosine.csv", "similarity_fib_spearman.csv",
    "similarity_stro_contrastive_cosine.csv",
    "correspondence_fib_by_cluster.csv", "correspondence_fib_by_subtype.csv",
    "correspondence_fib_by_subtype_spearman.csv", "correspondence_stro_by_subtype.csv",
    "TABLE_correspondence_summary.csv",
    "replication_per_sample.csv", "replication_per_phase.csv", "replication_consistency.csv",
    "robustness_marker_ablation.csv", "robustness_gene_bootstrap.csv", "robustness_heldout_split.csv",
    "marker_sets.csv", "headline_marker_fraction_positive.csv", "score_by_cluster.csv",
    "per_cell_subtype_scores.csv", "marker_panel_detectability.csv", "program_detectability.csv",
    "run_config.txt", "fig7_detectable_markers_dotplot.png",
]:
    INPUTS[f"human/{name}"] = OUT / name
for name in [
    "TABLE_correspondence_mouse.csv", "TABLE_correspondence_mouse_by_query.csv",
    "matrix_contrastive_cosine.csv", "matrix_raw_spearman.csv",
    "gene_bootstrap_mouse.csv", "heldout_mouse.csv", "meta_mouse.json",
    "hammoud_gene_drivers_v5.json",
]:
    INPUTS[f"mouse/{name}"] = MOUT / name
INPUTS["figures/dotplot__fib_published_markers.png"] = HAM / "figures" / "dotplot__fib_published_markers.png"
INPUTS["figures/dotplot__fib_negative_controls.png"] = HAM / "figures" / "dotplot__fib_negative_controls.png"
INPUTS["deliverable XMens h5ad (crosswalk only)"] = H5MM

rows = []
for label, p in INPUTS.items():
    ex = p.exists()
    st = p.stat() if ex else None
    rows.append({"input": label, "exists": ex,
                 "mtime": pd.Timestamp(st.st_mtime, unit="s").strftime("%Y-%m-%d %H:%M") if ex else "-",
                 "size": st.st_size if ex else 0})
manifest = pd.DataFrame(rows)
display(manifest)
verify("all input files present", int((~manifest["exists"]).sum()), 0, tol=0)

cfg = dict(l.strip().split("=", 1) for l in open(OUT / "run_config.txt") if "=" in l)
print("human-arm run config:", cfg)
verify("human arm: nuclei in run_config", int(cfg["n_obs"]), 47533, tol=0)

## 1. Method — why contrastive cosine is the headline, and proof the choice matters

The primary statistic is a **cluster-level centroid comparison** (the method the atlas paper itself
uses to reconcile annotations; its Fig. 3A is a 39×39 centroid rank-correlation matrix), with two
deliberate choices, unchanged from the human-arm deliverable:

- **Cluster centroids, not per-cell correlation, and no kNN smoothing.** Centroids are
  `log1p(mean(expm1(x)))` — averaged in linear CP10K space, then logged.
- **Double-centred ("contrastive") cosine as headline; raw Spearman shown for continuity.**

If the two metrics picked the same partners everywhere, the choice would be immaterial and should be
said so. It is not: the cell below verifies that they disagree on the best partner for **5 of 6**
atlas subtypes in the human arm and **6 of 9** query clusters in the mouse arm.

In [ ]:
bys   = pd.read_csv(OUT / "correspondence_fib_by_subtype.csv", index_col=0)
bysp  = pd.read_csv(OUT / "correspondence_fib_by_subtype_spearman.csv", index_col=0)
metric_cmp = pd.DataFrame({
    "best cluster (contrastive cosine)": bys["best_query"].astype(str),
    "best cluster (raw Spearman)": bysp["best_query"].astype(str),
})
metric_cmp.index = [SHORT.get(i, i) for i in metric_cmp.index]
metric_cmp["differ"] = metric_cmp.iloc[:, 0] != metric_cmp.iloc[:, 1]
display(metric_cmp)
verify("human: subtypes where Spearman and cosine pick different clusters", int(metric_cmp["differ"].sum()), 5, tol=0)

Mc_mm = pd.read_csv(MOUT / "matrix_contrastive_cosine.csv", index_col=0)
Mr_mm = pd.read_csv(MOUT / "matrix_raw_spearman.csv", index_col=0)
mm_cmp = pd.DataFrame({"cosine argmax (39 cols)": Mc_mm.idxmax(axis=1),
                       "Spearman argmax (39 cols)": Mr_mm.idxmax(axis=1)})
mm_cmp["differ"] = mm_cmp.iloc[:, 0] != mm_cmp.iloc[:, 1]
display(mm_cmp)
verify("mouse: query clusters where the two metrics pick different subtypes", int(mm_cmp["differ"].sum()), 6, tol=0)

## 2. HUMAN ARM — six atlas fibroblast subtypes vs our six leiden-0.5 clusters

Everything in this section is read from `fib_centroid_results/` (written by `run_mapping.py`,
`resolution_stability.py`, `why_no_match.py` on 2026-08-30). Nothing is recomputed from the count
matrix here.

In [ ]:
rd = lambda name, **kw: pd.read_csv(OUT / name, **kw)
M_cos  = rd("similarity_fib_contrastive_cosine.csv", index_col=0);  M_cos.index  = M_cos.index.astype(str)
M_spr  = rd("similarity_fib_spearman.csv", index_col=0);            M_spr.index  = M_spr.index.astype(str)
M_stro = rd("similarity_stro_contrastive_cosine.csv", index_col=0); M_stro.index = M_stro.index.astype(str)
by_cluster      = rd("correspondence_fib_by_cluster.csv", index_col=0)
by_subtype      = rd("correspondence_fib_by_subtype.csv", index_col=0)
by_subtype_spr  = rd("correspondence_fib_by_subtype_spearman.csv", index_col=0)
by_subtype_stro = rd("correspondence_stro_by_subtype.csv", index_col=0)
sizes = rd("cluster_sizes.csv", index_col=0)["n"]; sizes.index = sizes.index.astype(str)
sweep = rd("resolution_sweep.csv")
stab  = rd("resolution_stability.csv")
TBL   = rd("TABLE_correspondence_summary.csv", index_col=0)
per_sample  = rd("replication_per_sample.csv")
per_phase   = rd("replication_per_phase.csv")
consistency = rd("replication_consistency.csv")
abl  = rd("robustness_marker_ablation.csv")
boot = rd("robustness_gene_bootstrap.csv")
held = rd("robustness_heldout_split.csv")
marker_sets = rd("marker_sets.csv")
frac_pos = rd("headline_marker_fraction_positive.csv", index_col=0); frac_pos.index = frac_pos.index.astype(str)
score_by_cluster = rd("score_by_cluster.csv", index_col=0); score_by_cluster.index = score_by_cluster.index.astype(str)
det  = rd("marker_panel_detectability.csv")
prog = rd("program_detectability.csv")
print(f"loaded: {M_cos.shape[0]} clusters x {M_cos.shape[1]} fibroblast subtypes; "
      f"{sizes.sum():,} nuclei in {len(sizes)} clusters")

### 2.1 Choice of clustering resolution

Resolution was **set deliberately to 0.5**, not auto-selected: maximising reciprocal matches picks
2.0, but the extra "matches" arrive at noise-level scores while every real margin shrinks as 6
clusters become 21+. The sweep and the full per-resolution stability table are below.

Two claims, each verified: the Stro-7 correspondence is **reciprocal at every resolution from 0.3 to
2.0**. A prior write-up also claimed it carries *the largest margin at each* resolution — that does
**not** reproduce from the stored table (largest at 3 of 5; Stro-8 leads at 0.3 and Stro-10 at 0.8).
Prose drift, reported here rather than repeated.

In [ ]:
display(sweep.round(3))
s7 = stab[stab["subtype"] == "Stro-7 C7"]
display(s7.round(3))
verify("Stro-7 reciprocal at all resolutions", int(s7["reciprocal"].sum()), 5, tol=0)

largest = stab.loc[stab.groupby("resolution")["margin"].idxmax(), ["resolution", "subtype"]]
display(largest)
verify("resolutions where Stro-7 has the largest margin (prior prose said all 5)",
       int((largest["subtype"] == "Stro-7 C7").sum()), 3, tol=0,
       note="drift: prior claim 'largest margin at each' is wrong")

recip = stab.groupby("resolution")["reciprocal"].sum().astype(int)
agree = int((recip.values == sweep.set_index("resolution").loc[recip.index, "n_subtypes_reciprocal"].values).sum())
verify("stability table agrees with sweep on reciprocal counts (5 resolutions)", agree, 5, tol=0)

### 2.2 Primary result — the correspondence matrix

Rows are our clusters, columns the six atlas fibroblast subtypes. Black outline = reciprocal best
match. The Spearman companion is shown because the paper's own Fig. 3A uses it; §1 verified that it
disagrees with the headline metric for 5 of 6 subtypes, which is expected across the
single-nucleus/single-cell boundary.

In [ ]:
def corr_heatmap(M, by_a, title, cbar_label, cmap="RdBu_r", center=True, ax=None, annot=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(1.1 * M.shape[1] + 4, 0.42 * M.shape[0] + 3))
    v = np.nanmax(np.abs(M.to_numpy())) if center else None
    kw = dict(vmin=-v, vmax=v) if center else {}
    im = ax.imshow(M.to_numpy(), aspect="auto", cmap=cmap, **kw)
    ax.set_xticks(range(M.shape[1]))
    ax.set_xticklabels([SHORT.get(c, c) for c in M.columns], rotation=45, ha="right", fontsize=10)
    ax.set_yticks(range(M.shape[0]))
    ax.set_yticklabels([f"{i}  (n={int(sizes.get(str(i), 0)):,})" for i in M.index], fontsize=9)
    recip = {(str(by_a.loc[s, "best_query"]), s) for s in by_a.index if by_a.loc[s, "reciprocal"]}
    for i, q in enumerate(M.index):
        for j, s in enumerate(M.columns):
            if annot:
                val = M.iloc[i, j]
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(val) > 0.55 * np.nanmax(np.abs(M.to_numpy())) else "0.2")
            if (str(q), s) in recip:
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, edgecolor="black", lw=2.5))
    ax.set_title(title, fontsize=12, pad=12)
    plt.colorbar(im, ax=ax, shrink=0.6, label=cbar_label)
    return ax

fig, axes = plt.subplots(1, 2, figsize=(19, 0.42 * M_cos.shape[0] + 4))
corr_heatmap(M_cos, by_subtype, "Contrastive cosine (headline)", "double-centred cosine", ax=axes[0])
corr_heatmap(M_spr, by_subtype_spr, "Raw Spearman (as in atlas Fig. 3A)", "rank correlation",
             center=False, cmap="viridis", ax=axes[1])
plt.tight_layout(); plt.show()

# the stored by-subtype table must be a pure argmax of the stored matrix
rec_best  = M_cos.idxmax(axis=0)
rec_score = M_cos.max(axis=0)
verify("stored best_query is the argmax of the stored matrix (6 subtypes)",
       int((rec_best.values.astype(str) == by_subtype["best_query"].astype(str).values).sum()), 6, tol=0)
verify("stored best scores match the matrix (max |diff|)",
       float((rec_score.values - by_subtype["score"].values).max()), 0, tol=1e-9)

### 2.3 The correspondence table (the human deliverable)

One row per atlas fibroblast subtype, including the ones we cannot claim. Calls are calibrated
against the five subtypes that **cannot** have a counterpart here (perivascular Stro-1/2/3, vascular
progenitor Stro-4, uSMC Stro-6 — our object is fibroblast/decidual only), not against a hand-picked
threshold.

Headlines, each verified below: **Stro-7 C7 → cluster 3** (score 0.343, margin 0.236, confident);
**Stro-10 dS3 → cluster 1** (0.389, 0.172, confident); **Stro-8 dS1 → cluster 5** (0.273, probable);
Stro-9 dS2 not resolved from dS1; Stro-5 and Stro-11 have no counterpart above the negative-control
band.

In [ ]:
display(TBL.round(3))

NEG = ["Stro-1_PV MYH11 FOXC1hi CTGFlo", "Stro-2_PV MYH11 FOXC1lo CTGFhi", "Stro-3_PV STEAP4",
       "Stro-4_Tissue resident vascular progenitor", "Stro-6_uSMC"]
NEG_CEILING = by_subtype_stro.loc[NEG, "score"].max()
verify("negative-control ceiling", float(NEG_CEILING), 0.1235, tol=0.01)
rec_above = (TBL["score"] > NEG_CEILING)
verify("above_neg_controls column reproduces from the ceiling (6 subtypes)",
       int((rec_above == TBL["above_neg_controls"]).sum()), 6, tol=0)

verify("Stro-7 C7 score",  float(TBL.loc["Stro-7 C7", "score"]),  0.343, note="cluster 3")
verify("Stro-7 C7 margin", float(TBL.loc["Stro-7 C7", "margin"]), 0.236)
verify("Stro-10 dS3 score",  float(TBL.loc["Stro-10 dS3", "score"]),  0.389, note="cluster 1")
verify("Stro-10 dS3 margin", float(TBL.loc["Stro-10 dS3", "margin"]), 0.172)
verify("Stro-8 dS1 score", float(TBL.loc["Stro-8 dS1", "score"]), 0.273, note="cluster 5")

# the stored call column must reproduce from the stored evidence columns + the recorded rule
owner = {r["best_query"]: s for s, r in TBL.iterrows() if r["reciprocal"]}
def call(r):
    if not r["above_neg_controls"]:
        return "no counterpart above negative controls"
    if not r["reciprocal"] and r["best_query"] in owner:
        return f"not resolved from {owner[r['best_query']]}"
    if (r["reciprocal"] and r["bootstrap_freq"] >= .9
            and r["heldout_agreement"] >= .9 and r["sample_agreement"] >= .8):
        return "confident"
    if r["reciprocal"] and r["bootstrap_freq"] >= .7 and r["heldout_agreement"] >= .7:
        return "probable"
    return "one-directional"
rec_calls = TBL.apply(call, axis=1)
verify("stored calls reproduce from the recorded rule (6 subtypes)",
       int((rec_calls == TBL["call"]).sum()), 6, tol=0)

### 2.4 Sink test — are the matches real, or argmax with nowhere else to go?

The same mapping run against **all eleven** stromal subtypes: the five structural negative controls
should find no good partner. They band at 0.081–0.124 and none is reciprocal; the two confident
calls sit at 2.8× and 3.2× that ceiling.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 0.42 * M_stro.shape[0] + 4))
corr_heatmap(M_stro, by_subtype_stro, "All 11 stromal subtypes (negative controls included)",
             "double-centred cosine", ax=ax, annot=False)
plt.tight_layout(); plt.show()

band = by_subtype_stro.loc[NEG, "score"]
verify("negative-control band, low",  float(band.min()), 0.0813, tol=0.01)
verify("negative-control band, high", float(band.max()), 0.1235, tol=0.01)
verify("negative controls that are reciprocal", int(by_subtype_stro.loc[NEG, "reciprocal"].sum()), 0, tol=0)
summary = by_subtype_stro.copy()
summary["is_negative_control"] = summary.index.isin(NEG)
display(summary.sort_values(["is_negative_control", "score"], ascending=[True, False]).round(3))

### 2.5 Replication across sections and phases

The mapping recomputed from scratch inside each of the six sections — the direct answer to the
two-donor comment. **Caveat stated up front:** the two donors are `mck5` (menstrual) and `mck6`
(secretory), so donor and cycle phase are perfectly confounded. Stro-7 and Stro-10 come back to the
same cluster in **6/6 sections**; Stro-5/8/9 agree in 3/6. For **Stro-8 and Stro-9** the 3/6 split
falls exactly on the phase boundary (one cluster in all three menstrual sections, a different one in
all three secretory), so their 0.5 may be phase-tracking rather than failed replication. **Stro-5**
is consistent only within the menstrual sections and scatters in secretory — plain instability, not
a phase effect.

In [ ]:
pivot = per_sample.pivot(index="subtype", columns="sample", values="best_query")
pivot.index = [SHORT.get(i, i) for i in pivot.index]
display(pivot)
display(per_phase.round(3))

rec_cons = per_sample.groupby("subtype")["best_query"].agg(
    modal=lambda s: s.value_counts().index[0],
    agreement=lambda s: s.value_counts(normalize=True).iloc[0],
    n_samples="size").reset_index()
merged = rec_cons.merge(consistency, on="subtype", suffixes=("_re", ""))
ok = int(((merged["modal_re"].astype(str) == merged["modal"].astype(str))
          & (np.isclose(merged["agreement_re"], merged["agreement"]))
          & (merged["n_samples_re"] == merged["n_samples"])).sum())
verify("stored consistency table reproduces from per-sample results (6 subtypes)", ok, 6, tol=0)
verify("Stro-7 section agreement",  float(consistency.set_index("subtype").loc["Stro-7_C7 Fibroblast", "agreement"]), 1.0, tol=0)
verify("Stro-10 section agreement", float(consistency.set_index("subtype").loc["Stro-10_dS3", "agreement"]), 1.0, tol=0)

# the phase-split reading of the 0.5 agreements, verified rather than asserted
pv = per_sample.pivot(index="subtype", columns="sample", values="best_query").astype(str)
men = [c for c in pv.columns if c.startswith("menstrual")]
sec = [c for c in pv.columns if c.startswith("secretory")]
def phase_split(subtype):
    row = pv.loc[subtype]
    return int(row[men].nunique() == 1 and row[sec].nunique() == 1 and row[men].iloc[0] != row[sec].iloc[0])
verify("Stro-8 3/6 split falls exactly on the phase boundary", phase_split("Stro-8_dS1"), 1, tol=0)
verify("Stro-9 3/6 split falls exactly on the phase boundary", phase_split("Stro-9_dS2"), 1, tol=0)
verify("Stro-5 split is NOT phase-consistent", phase_split("Stro-5_RUNX3+ Fibroblast"), 0, tol=0)
display(consistency.round(3))

### 2.6 Robustness — marker ablation, gene bootstrap, held-out genes

All three resample **genes** on the cached centroid matrix. Verified headlines: Stro-7 and Stro-10
keep the same cluster in 100% of 500 bootstraps and 100% of held-out splits; Stro-5's held-out
agreement is 0.024 (its apparent match does not survive genes it was not selected on); no subtype's
ablation margin ever crosses zero.

In [ ]:
display(boot.round(3)); display(held.round(3)); display(abl.round(3))
b = boot.set_index("subtype"); h = held.set_index("subtype")
verify("Stro-7 bootstrap frequency",  float(b.loc["Stro-7_C7 Fibroblast", "frequency"]), 1.0, tol=0)
verify("Stro-10 bootstrap frequency", float(b.loc["Stro-10_dS3", "frequency"]), 1.0, tol=0)
verify("Stro-7 held-out agreement",   float(h.loc["Stro-7_C7 Fibroblast", "validation_agreement"]), 1.0, tol=0)
verify("Stro-10 held-out agreement",  float(h.loc["Stro-10_dS3", "validation_agreement"]), 1.0, tol=0)
verify("Stro-5 held-out agreement (fails validation)", float(h.loc["Stro-5_RUNX3+ Fibroblast", "validation_agreement"]), 0.024, tol=0)
verify("smallest ablation margin across all subtypes stays positive", int(float(abl["margin"].min()) > 0), 1, tol=0)

### 2.7 Marker-level evidence, and prose drift found during this compilation

Centroid correlations *nominate* pairs; what carries a rebuttal is whether the defining markers are
expressed in both populations. Two marker sources are compared (atlas-centroid-derived vs the
paper's own published DE markers); their overlap is itself a check. The dotplots are the stored
outputs of the prior run, embedded unchanged.

In [ ]:
display(marker_sets[["subtype", "n_centroid_derived", "n_published_detected", "n_shared"]])
ns_ = marker_sets.set_index("subtype")["n_shared"]
verify("Stro-7 marker sources overlap (n shared of 50)", int(ns_["Stro-7_C7 Fibroblast"]), 35, tol=0)
verify("Stro-5 marker sources overlap (n shared of 50)", int(ns_["Stro-5_RUNX3+ Fibroblast"]), 45, tol=0)
display(Image(filename=str(HAM / "figures" / "dotplot__fib_published_markers.png"), width=950))
display(Image(filename=str(HAM / "figures" / "dotplot__fib_negative_controls.png"), width=950))

#### The C7 block — recomputed from the artifacts, with the prior prose checked against them

C7 itself is sparse in single-nucleus data, so the prior write-up quantified its enrichment in the
nominated cluster. Recomputing every one of those numbers from the stored per-cluster detection
fractions and cluster sizes: **C7 is detected in 0.90% of nuclei object-wide, 5.60% in cluster 3 vs
0.39% elsewhere (14.4×), and cluster 3 holds 61.0% of all C7+ nuclei while being 9.8% of the
object.** The centroid mapping nominated cluster 3 without reference to C7's sparsity, so low
absolute detection is a property of snRNA, not weak evidence.

**Drift report.** Four of the six numbers in the prior notebook's version of this paragraph do not
reproduce from the stored artifacts (table below). The direction is mixed (16.2×→14.4× overstated;
58.2%→61.0% understated), which is consistent with them having been computed against a different,
unrecorded clustering vintage rather than embellished. The artifacts are authoritative; the values
above are the recomputed ones, and the prior values should not be quoted again.

In [ ]:
g = "C7"
c7_obj  = float((frac_pos[g] * sizes).sum() / sizes.sum())
c7_in3  = float(frac_pos.loc["3", g])
c7_else = float((frac_pos[g] * sizes).drop("3").sum() / sizes.drop("3").sum())
c7_ratio = c7_in3 / c7_else
c7_share = float(frac_pos.loc["3", g] * sizes["3"] / (frac_pos[g] * sizes).sum())
c7_size  = float(sizes["3"] / sizes.sum())
verify("C7 detected object-wide",              c7_obj,   0.0090, tol=0.02)
verify("C7 detected in cluster 3",             c7_in3,   0.0560, tol=0.02)
verify("C7 detected elsewhere",                c7_else,  0.0039, tol=0.02)
verify("C7 enrichment in cluster 3",           c7_ratio, 14.4,   tol=0.02)
verify("share of C7+ nuclei held by cluster 3", c7_share, 0.610,  tol=0.02)
verify("cluster 3 share of the object",        c7_size,  0.098,  tol=0.02)

drift = pd.DataFrame({
    "quantity": ["object-wide detection", "cluster-3 detection", "elsewhere detection",
                 "enrichment ratio", "share of C7+ nuclei", "cluster-3 share of object"],
    "prior prose": ["0.90%", "6.63%", "0.41%", "16.2x", "58.2%", "7.9%"],
    "recomputed from artifacts": [f"{c7_obj:.2%}", f"{c7_in3:.2%}", f"{c7_else:.2%}",
                                  f"{c7_ratio:.1f}x", f"{c7_share:.1%}", f"{c7_size:.1%}"],
    "reproduces": ["yes", "NO", "NO (5% off)", "NO", "NO", "NO"],
})
print("PROSE-DRIFT REPORT — prior C7 paragraph vs stored artifacts:")
display(drift)

In [ ]:
# Marker-set scores per cluster (stored) must reproduce from the stored per-cell scores.
per_cell = pd.read_csv(OUT / "per_cell_subtype_scores.csv", index_col=0)
sc_cols = [c for c in per_cell.columns if c.startswith("score_")]
rec_sbc = per_cell.groupby("leiden_0.5")[sc_cols].mean()
rec_sbc.index = rec_sbc.index.astype(str)
maxdiff = float((rec_sbc - score_by_cluster).abs().max().max())
verify("score_by_cluster reproduces exactly from the per-cell scores", int(maxdiff < 1e-9), 1, tol=0)
display(score_by_cluster.round(3))
verify("cluster 3 tops the Stro-7 score column",  score_by_cluster["score_Stro-7"].idxmax(),  "3", tol=0)
verify("cluster 1 tops the Stro-10 score column", score_by_cluster["score_Stro-10"].idxmax(), "1", tol=0)
# honesty check, not a headline: the Stro-8 score peaks in cluster 4, NOT the nominated cluster 5.
verify("Stro-8 score peaks in cluster 4, not the nominated cluster 5 (stated, not hidden)",
       score_by_cluster["score_Stro-8"].idxmax(), "4", tol=0)

### 2.8 Why three subtypes cannot map — detectability, quantified

Each non-match is defined by a program our object excludes by construction or measures differently.
Verified: the two "no counterpart" subtypes (Stro-5, Stro-11) have the two **lowest** marker-panel
detection rates; only 0.98% of nuclei are positive for any immune-panel gene (Stro-5's definition),
2.9% for any cycling gene (Stro-11's), and MKI67 is detected in 0.006% of nuclei.

In [ ]:
display(det.round(4)); display(prog.round(4))
two_lowest = set(det.nsmallest(2, "mean_detection_rate")["subtype"])
verify("two lowest detection rates are the two no-counterpart subtypes",
       int(two_lowest == {"Stro-5 RUNX3+", "Stro-11 progenitor"}), 1, tol=0)
p = prog.set_index("program")["nuclei_positive_for_any"]
verify("nuclei positive for any immune-panel gene",  float(p["immune (defines Stro-5)"]),  0.0098, tol=0.02)
verify("nuclei positive for any cycling-panel gene", float(p["cycling (defines Stro-11)"]), 0.0293, tol=0.02)
mki = float((frac_pos["MKI67"] * sizes).sum() / sizes.sum())
verify("MKI67 detected object-wide", mki, 0.000063, tol=0.05)
display(Image(filename=str(OUT / "fig7_detectable_markers_dotplot.png"), width=900))

### 2.9 Per-cell layer and spatial readout

A depth-corrected **marker-set score** (score_genes control-gene binning, no kNN smoothing),
re-rendered here directly from the stored per-cell table — read it as a continuous program score,
not a per-cell identity call. Diverging scale centred at zero.

In [ ]:
tps = sorted(per_cell["timepoint"].astype(str).unique())
fig, axes = plt.subplots(len(sc_cols), len(tps),
                         figsize=(4.2 * len(tps), 3.6 * len(sc_cols)), squeeze=False)
x = per_cell["x_um_dbscan"].to_numpy(float); y = per_cell["y_um_dbscan"].to_numpy(float)
for i, col in enumerate(sc_cols):
    v = per_cell[col].to_numpy(float)
    lim = np.nanpercentile(np.abs(v), 99)
    for j, tp in enumerate(tps):
        m = (per_cell["timepoint"].astype(str) == tp).to_numpy() & np.isfinite(x) & np.isfinite(y)
        o = np.argsort(np.abs(v[m]))
        ax = axes[i, j]
        pcm = ax.scatter(x[m][o], y[m][o], c=v[m][o], s=2.5, cmap="RdBu_r",
                         vmin=-lim, vmax=lim, linewidths=0, rasterized=True)
        ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_axis_off()
        if i == 0: ax.set_title(tp, fontsize=12)
        if j == 0:
            ax.text(-0.05, .5, col.replace("score_", ""), transform=ax.transAxes,
                    rotation=90, va="center", ha="right", fontsize=11)
    fig.colorbar(pcm, ax=axes[i, :].tolist(), shrink=.75)
plt.show()

## 3. MOUSE ARM (XMens) — new; not in any prior notebook

Run 2026-08-30 via `run_mapping_mouse.py`: same module, same statistics. Differences from the human
arm, and only these: (1) the atlas gene axis was translated through **Ensembl 116 strict one-to-one
orthologs** (15,681 of 33,791 atlas genes survive; 15,606 shared with the object); (2) groups are
the curated `fine_refined` labels — **no clustering step**, so the resolution machinery of §2.1 is
**N/A by design** here; (3) the saved matrices score against **all 39 atlas subtypes** and use the
**full 15,606-gene shared panel** — no variance panel — so their double-centring spans 39 columns
and their magnitudes are not comparable to the human §2 matrix. Driver-JSON cosines (§4) use a
2,728-gene discriminative panel and are the panel-matched magnitudes.

Gene symbols throughout the mouse arm are **uppercase human symbols**: the mouse panel was mapped
through orthologs into human symbol space. That is correct, but worth saying once.

### 3.0 Crosswalk first — three vocabularies name the same nine clusters

Three files use three different names for the mouse clusters (short codes, driver-JSON full names,
h5ad `cell_subtype`). An earlier wrong claim in this project came from exactly this class of
mismatch, so the crosswalk is verified against the deliverable h5ad itself before anything is joined.

In [ ]:
CROSSWALK = {   # short code (results CSVs)  ->  canonical full name (driver JSON == h5ad cell_subtype)
    "dec1":  "decidual_boundary",
    "dec2":  "decidual_proliferative",
    "dec3":  "decidual_mature",
    "decAp": "decidual_apoptotic",
    "eFib1": "efib_outer",
    "eFib2": "efib_middle",
    "eFib2i":"efib_middle_induced",
    "eFib3": "efib_subluminal",
    "eFibP": "efib_proliferative",
}
import h5py
with h5py.File(H5MM, "r") as f:
    fr = pd.Categorical.from_codes(f["obs/fine_refined/codes"][:],
                                   [c.decode() for c in f["obs/fine_refined/categories"][:]])
    cs = pd.Categorical.from_codes(f["obs/cell_subtype/codes"][:],
                                   [c.decode() for c in f["obs/cell_subtype/categories"][:]])
ct = pd.crosstab(fr, cs)
cross_tbl = pd.DataFrame({
    "short_code": ct.index,
    "cell_subtype_in_h5ad": [ct.loc[c].idxmax() for c in ct.index],
    "n_cells": [int(ct.loc[c].sum()) for c in ct.index],
    "maps_1to1": [int((ct.loc[c] > 0).sum()) == 1 for c in ct.index],
}).set_index("short_code")
display(cross_tbl)
verify("every short code maps to exactly one cell_subtype (9 clusters)", int(cross_tbl["maps_1to1"].sum()), 9, tol=0)
verify("hard-coded crosswalk matches the h5ad (9 clusters)",
       int(sum(cross_tbl.loc[c, "cell_subtype_in_h5ad"] == CROSSWALK[c] for c in CROSSWALK)), 9, tol=0)

bq = pd.read_csv(MOUT / "TABLE_correspondence_mouse_by_query.csv", index_col=0)
ba = pd.read_csv(MOUT / "TABLE_correspondence_mouse.csv", index_col=0)
verify("cell counts agree: h5ad vs by-query table (9 clusters)",
       int(sum(int(cross_tbl.loc[c, "n_cells"]) == int(bq.loc[c, "n"]) for c in CROSSWALK)), 9, tol=0)

DRIVERS = json.loads((MOUT / "hammoud_gene_drivers_v5.json").read_text())
mouse_json = {}
for k, v in DRIVERS.items():
    sp, rest = k.split(":", 1)
    q, a = rest.split("->", 1)
    if sp == "mouse":
        mouse_json[q] = dict(v, atlas=a)
verify("cell counts agree: h5ad vs driver JSON (9 mouse queries)",
       int(sum(int(cross_tbl.loc[c, "n_cells"]) == int(mouse_json[CROSSWALK[c]]["n_cells"]) for c in CROSSWALK)), 9, tol=0)

meta = json.loads((MOUT / "meta_mouse.json").read_text())
print(meta)
verify("mouse object size (meta)", int(meta["n_cells"]), 43197, tol=0)
verify("shared ortholog panel (meta)", int(meta["panel_genes"]), 15606, tol=0)
verify("atlas genes surviving strict 1:1 translation (meta)", int(meta["atlas_genes_translated"]), 15681, tol=0)
verify("by-query cell counts sum to the object", int(bq["n"].sum()), 43197, tol=0)

### 3.1 Primary mouse table — re-argmax of the saved matrix, restricted to the six fibroblast subtypes

The saved by-query table picks its best match over **all 39** atlas subtypes, which sends stromal
queries to glandular and immune subtypes (shown in §3.2 as the specificity check). The primary
presentation here restricts the **saved** similarity matrix to the six fibroblast Stro subtypes and
re-takes the argmax — a pure read, no refit. Per-row negative-control ceiling = each query's best
score among the five structural negative controls (Stro-1/2/3/4/6), the §2.4 logic applied row-wise.

Verified headlines: **efib_outer → Stro-7 C7** (0.174 full panel / 0.243 driver panel) and
**efib_proliferative → Stro-11** (0.259 / 0.347) clear their ceilings decisively;
**efib_middle_induced is UNRESOLVED** (Stro-5 vs Stro-10 gap 0.0017 — reported as neither);
**decidual_mature and decidual_apoptotic fall below their negative-control ceilings on the full
panel** — decidual_mature is panel-dependent (0.255 on the discriminative panel, §4's strongest
credible decidual signal), decidual_apoptotic is a housekeeping artifact (§4). The restricted
argmax agrees with the driver JSON's target for all 8 comparable clusters.

In [ ]:
Mc_mm.index = Mc_mm.index.astype(str)
NEG5 = ["Stro-1_PV MYH11 FOXC1hi CTGFlo", "Stro-2_PV MYH11 FOXC1lo CTGFhi", "Stro-3_PV STEAP4",
        "Stro-4_Tissue resident vascular progenitor", "Stro-6_uSMC"]
F = Mc_mm[FIB_SUBTYPES]
prim = pd.DataFrame({
    "canonical_name": [CROSSWALK[c] for c in F.index],
    "best_fib_subtype": F.idxmax(axis=1),
    "score_full_panel": F.max(axis=1),
    "margin_within_6": F.apply(lambda r: r.nlargest(2).iloc[0] - r.nlargest(2).iloc[1], axis=1),
    "neg_ctrl_ceiling": Mc_mm[NEG5].max(axis=1),
})
prim["above_ceiling"] = prim["score_full_panel"] > prim["neg_ctrl_ceiling"]
prim["json_target"] = [mouse_json[CROSSWALK[c]]["atlas"] for c in F.index]
prim["json_panel_cosine"] = [mouse_json[CROSSWALK[c]]["cosine"] for c in F.index]

def mouse_call(r):
    if r["margin_within_6"] < 0.005:
        return "UNRESOLVED (margin < 0.005)"
    if not r["above_ceiling"]:
        return "no match on full panel"
    if r["score_full_panel"] >= 0.05:
        return "match (full panel)"
    return "panel-dependent"
prim["call"] = prim.apply(mouse_call, axis=1)
display(prim.round(4))

verify("efib_middle_induced margin below the 0.005 floor -> UNRESOLVED",
       int(float(prim.loc["eFib2i", "margin_within_6"]) < 0.005), 1, tol=0)
comparable = [c for c in prim.index if c != "eFib2i"]
verify("restricted argmax agrees with the driver JSON target (8 comparable clusters)",
       int(sum(prim.loc[c, "best_fib_subtype"] == prim.loc[c, "json_target"] for c in comparable)), 8, tol=0)
verify("queries below their negative-control ceiling", int((~prim["above_ceiling"]).sum()), 2, tol=0,
       note="decidual_mature (dec3) and decidual_apoptotic (decAp)")
verify("efib_outer -> Stro-7 score (full panel)", float(prim.loc["eFib1", "score_full_panel"]), 0.1744)
verify("efib_outer -> Stro-7 cosine (driver panel)", float(prim.loc["eFib1", "json_panel_cosine"]), 0.2429)
verify("efib_proliferative -> Stro-11 score (full panel)", float(prim.loc["eFibP", "score_full_panel"]), 0.2593)
verify("decidual_mature best-fib score is NEGATIVE on the full panel", int(float(prim.loc["dec3", "score_full_panel"]) < 0), 1, tol=0)
verify("decidual_mature -> Stro-8 cosine on the driver panel", float(prim.loc["dec3", "json_panel_cosine"]), 0.2553)

fig, ax = plt.subplots(figsize=(10, 7))
v = np.nanmax(np.abs(F.to_numpy()))
im = ax.imshow(F.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v)
ax.set_xticks(range(F.shape[1])); ax.set_xticklabels([SHORT.get(c, c) for c in F.columns], rotation=45, ha="right")
ax.set_yticks(range(F.shape[0]))
ax.set_yticklabels([f"{CROSSWALK[i]} ({i}, n={int(bq.loc[i,'n']):,})" for i in F.index], fontsize=9)
for i in range(F.shape[0]):
    for j in range(F.shape[1]):
        ax.text(j, i, f"{F.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(F.iloc[i, j]) > 0.55 * v else "0.2")
ax.set_title("Mouse clusters vs the six fibroblast subtypes\n"
             "(saved matrix: full 15,606-gene panel, 39-column centring)", fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.7, label="double-centred cosine")
plt.tight_layout(); plt.show()

### 3.2 The all-39 tables — presented once, as a specificity check

Offered the whole atlas, **6 of 9 stromal queries pick a non-stromal subtype** (glandular,
proliferative-epithelial, or B cells). That is itself a finding worth stating: at this
cross-species, cross-platform distance, the full-panel signal does not preferentially recover
lineage. It is also why the all-39 `best_query` column (e.g. endothelial subtypes "matching"
decidual_apoptotic) is **not interpretable as a mouse result** and appears only here.

In [ ]:
lineage = lambda s: ("stromal" if s.startswith("Stro-") else
                     "immune" if s.startswith("Imm-") or s == "9_Tissue Res Mem T cell" else
                     "endothelial" if s.startswith(("Endo", "L_Endo")) else "epithelial/glandular")
bq_show = bq.copy()
bq_show.insert(0, "canonical_name", [CROSSWALK[c] for c in bq_show.index])
bq_show["best_atlas_lineage"] = [lineage(s) for s in bq_show["best_atlas"]]
display(bq_show.round(4))
verify("queries whose all-39 best match is non-stromal", int((bq_show["best_atlas_lineage"] != "stromal").sum()), 6, tol=0)

n_decAp_best = int((ba["best_query"] == "decAp").sum())
print(f"atlas subtypes whose best query is decidual_apoptotic (n=907, the smallest cluster): {n_decAp_best} of 39")
fig, ax = plt.subplots(figsize=(16, 4.5))
v = np.nanmax(np.abs(Mc_mm.to_numpy()))
im = ax.imshow(Mc_mm.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v)
ax.set_xticks(range(Mc_mm.shape[1])); ax.set_xticklabels(Mc_mm.columns, rotation=90, fontsize=7)
ax.set_yticks(range(Mc_mm.shape[0])); ax.set_yticklabels([CROSSWALK[i] for i in Mc_mm.index], fontsize=9)
ax.set_title("Specificity check: mouse clusters vs all 39 atlas subtypes (full panel)", fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.8, label="double-centred cosine")
plt.tight_layout(); plt.show()

### 3.3 Robustness that exists for the mouse arm — gene bootstrap and held-out genes

Run with the full atlas as competitors (39 columns) and the six fibroblast subtypes as targets.
Verified: Stro-7 recovers efib_outer in 100% of bootstraps and 99.8% of held-out splits; Stro-11
recovers efib_proliferative in 100% and 98.8%. Stro-8 is split (efib_middle_induced 66%,
decidual_apoptotic 33% — consistent with §3.1's finding that its full-panel signal is weak), and
Stro-5/Stro-9's held-out agreement collapses (0.024 / 0.13), so their bootstrap modes are not
evidence.

In [ ]:
boot_mm = pd.read_csv(MOUT / "gene_bootstrap_mouse.csv", index_col=0).set_index("subtype")
held_mm = pd.read_csv(MOUT / "heldout_mouse.csv", index_col=0).set_index("subtype")
display(boot_mm.round(3)); display(held_mm.round(3))
verify("Stro-7 bootstrap: modal query is efib_outer (eFib1)", boot_mm.loc["Stro-7_C7 Fibroblast", "modal_query"], "eFib1", tol=0)
verify("Stro-7 bootstrap frequency",  float(boot_mm.loc["Stro-7_C7 Fibroblast", "frequency"]), 1.0, tol=0)
verify("Stro-11 bootstrap: modal query is efib_proliferative (eFibP)", boot_mm.loc["Stro-11_Stromal progenitor", "modal_query"], "eFibP", tol=0)
verify("Stro-11 bootstrap frequency", float(boot_mm.loc["Stro-11_Stromal progenitor", "frequency"]), 1.0, tol=0)
verify("Stro-7 held-out agreement",   float(held_mm.loc["Stro-7_C7 Fibroblast", "validation_agreement"]), 0.998, tol=0)
verify("Stro-11 held-out agreement",  float(held_mm.loc["Stro-11_Stromal progenitor", "validation_agreement"]), 0.988, tol=0)
verify("Stro-8 bootstrap is split (modal frequency)", float(boot_mm.loc["Stro-8_dS1", "frequency"]), 0.662, tol=0)
verify("Stro-5 held-out agreement collapses", float(held_mm.loc["Stro-5_RUNX3+ Fibroblast", "validation_agreement"]), 0.024, tol=0)

### 3.4 What was NEVER RUN for the mouse arm — explicit, not a silent gap

These §2 analyses have no mouse counterpart because they were **never run** — not because they
failed. Decision on file (2026-09-01): leave them un-run. The resolution machinery (§2.1) is a
separate case: **N/A by design**, because the mouse arm used curated labels and never clustered.

In [ ]:
never_run = pd.DataFrame([
    ("replication per section / per timepoint", "replication_per_sample.csv etc.", "NOT RUN"),
    ("replication consistency",                 "replication_consistency.csv",     "NOT RUN"),
    ("marker ablation",                         "robustness_marker_ablation.csv",  "NOT RUN"),
    ("marker sets + dotplots",                  "marker_sets.csv, figures",        "NOT RUN"),
    ("per-cell scores + spatial readout",       "per_cell_subtype_scores.csv",     "NOT RUN"),
    ("marker/program detectability",            "*_detectability.csv",             "NOT RUN"),
    ("resolution sweep + stability",            "resolution_*.csv",                "N/A by design (curated labels)"),
], columns=["analysis", "human artifact", "mouse status"])
display(never_run)
ghosts = ["replication_per_sample_mouse.csv", "replication_consistency_mouse.csv",
          "robustness_marker_ablation_mouse.csv", "marker_sets_mouse.csv",
          "per_cell_subtype_scores_mouse.csv", "program_detectability_mouse.csv"]
verify("none of the never-run analyses left partial outputs", int(sum((MOUT / g).exists() for g in ghosts)), 0, tol=0)

## 4. GENE DRIVERS — what genes carry each correspondence

From `hammoud_gene_drivers_v5.json` (exported to the cluster 2026-09-01; **do not regenerate** — the
computation failed three times before it worked). 19 correspondences: 9 mouse, 10 human.

Three provenance facts before any interpretation:

1. **Panel.** JSON cosines were computed on a discriminative panel (2,728 genes mouse / 3,099
   human), NOT on the saved matrices' panels. Where both exist, both are shown (§3.1); they are
   different quantities and are each verified against their own file.
2. **The human queries h0–h9 are a 10-cluster vintage whose mapping to the §2 leiden-0.5 clusters
   is UNKNOWN** (their cell counts sum to the full 47,533-nucleus object, so it is a different
   clustering). No stored file links them. Every cross-species comparison below is therefore keyed
   on **Stro subtype only**, never on "our cluster N". If the vintage is identified later this is a
   one-line amendment.
3. **Symbols are uppercase human symbols in both species** (mouse genes were mapped through
   orthologs into human symbol space).

Each driver gene is classified **identity / housekeeping / cell_cycle** (housekeeping by symbol
prefix — ribosomal, translation, respiratory; cell-cycle by a mitotic gene list). A match whose
drivers are not identity genes is not evidence of conserved cell identity.

In [ ]:
HK_RE = re.compile(r"^(RPS|RPL|MRP|EEF|EIF|TMSB|UQCR|ROMO|SEC61|SERP(?!IN)|PSM|NDUF|COX|ATP5)")
CC_SET = set("""MKI67 TOP2A CENPF UBE2C BIRC5 CDK1 CDKN3 ZWINT GINS2 TFDP1 NCAPH2 DIAPH3 G2E3
MXD3 PHF19 PSRC1 ASPM KIF23 FBXO5 ESCO2 DEPDC1 KNSTRN TROAP STMN1 LMNB1 CKAP2L HMGB2 TUBA1B
HIST1H4C CCNB1 CCNA2 CCNE1 PCNA TYMS RRM2 TK1 AURKA AURKB PLK1 BUB1 BUB1B CDC20 KIF11 KIF2C
KIF20A KIF4A NUSAP1 PRC1 RACGAP1 SPC24 SPC25 NDC80 NUF2 TPX2 ANLN ECT2 FOXM1 E2F1 MYBL2""".split())
CC_RE = re.compile(r"^(CENP|MCM\d|GINS\d|CDCA\d|HIST\d)")
def driver_class(g):
    if g in CC_SET or CC_RE.match(g): return "cell_cycle"
    if HK_RE.match(g): return "housekeeping"
    return "identity"

INV = {v: k for k, v in CROSSWALK.items()}
drows = []
for k, v in DRIVERS.items():
    sp, rest = k.split(":", 1)
    q, a = rest.split("->", 1)
    genes = [g for g, _ in v["top15"]]
    cls = pd.Series([driver_class(g) for g in genes]).value_counts()
    drows.append({
        "species": sp, "query": q, "short_code": INV.get(q, "(vintage unknown)"),
        "atlas_subtype": SHORT.get(a, a), "cosine": v["cosine"],
        "n_cells": v["n_cells"], "n_panel": v["n_panel"], "frac_from_top30": v["frac_from_top30"],
        "n_identity": int(cls.get("identity", 0)), "n_housekeeping": int(cls.get("housekeeping", 0)),
        "n_cell_cycle": int(cls.get("cell_cycle", 0)), "top15": ", ".join(genes),
    })
DRV = pd.DataFrame(drows).sort_values(["species", "atlas_subtype", "cosine"], ascending=[True, True, False])
display(DRV.drop(columns="top15").round(4))
with pd.option_context("display.max_colwidth", 200):
    display(DRV[["species", "query", "atlas_subtype", "top15"]])

verify("driver JSON entries", len(DRV), 19, tol=0)
verify("mouse entries", int((DRV["species"] == "mouse").sum()), 9, tol=0)
verify("human entries", int((DRV["species"] == "human").sum()), 10, tol=0)
verify("strongest correspondence overall (human h5 -> Stro-7)", float(DRV["cosine"].max()), 0.3671, tol=0)
verify("weakest (human h3 -> Stro-10, NEGATIVE: an argmax over nothing)", float(DRV["cosine"].min()), -0.0074, tol=0)

#### 4a. `frac_from_top30` is a ratio, NOT a fraction

Its generating code is not on the cluster (the export came from the artifact store), but its values
show what it is: the share of the cosine carried by the atlas's own top-30 markers, as a ratio whose
denominator is the total cosine. With a near-zero denominator it explodes: **1.519** for
decidual_apoptotic (cosine 0.019) and **−3.856** for h3 (cosine −0.007). Verified below; those two
rows are excluded from interpretation, and the column must never be labelled a percentage.

Where the denominator is healthy, the pattern is meaningful: the strong matches are carried mostly
by the *wider* panel (h5→Stro-7: 0.13; efib_outer→Stro-7: 0.22), i.e. they do not hang on a handful
of headline markers — consistent with the §2.6 ablation result.

In [ ]:
verify("frac_from_top30 for decidual_apoptotic exceeds 1", float(DRV.set_index("query").loc["decidual_apoptotic", "frac_from_top30"]), 1.519, tol=0)
verify("frac_from_top30 for h3 is negative", float(DRV.set_index("query").loc["h3", "frac_from_top30"]), -3.856, tol=0)
excl = DRV[DRV["cosine"].abs() < 0.05][["species", "query", "atlas_subtype", "cosine", "frac_from_top30"]]
print("rows excluded from frac_from_top30 interpretation (|cosine| < 0.05, ratio unstable):")
display(excl.round(4))

#### 4b. Drivers shared between the human and mouse routes to the SAME Stro subtype

Genes on both species' top-15 lists for the same subtype are **three-way conserved**
(atlas ↔ human ↔ mouse) — the strongest single claim in this body of work. Exclusions, stated:
`decidual_apoptotic` (housekeeping artifact, §4c) and `h3` (negative cosine) contribute no lists.
Stro-11 has no human route at all (no human cluster maps to it), so no comparison exists there.

Verified: **Stro-7 C7 — 6 shared identity drivers** (APOD, ITGA11, PTGIS, RERG, SCN7A, TNXB);
**Stro-8 dS1 — 2** (PALMD, SHOC2); **Stro-10 dS3 — 11** (APCDD1, CDH11, HDDC2, KAT6B, LMO4, NR2F2,
SEZ6L2, SMAD5, TMEM219, WNT2, ZNF516).

In [ ]:
top15 = {k: {g for g, _ in v["top15"]} for k, v in DRIVERS.items()}
def route_union(species, subtype, exclude=()):
    out = set()
    for k, v in DRIVERS.items():
        sp, rest = k.split(":", 1); q, a = rest.split("->", 1)
        if sp == species and a == subtype and q not in exclude:
            out |= top15[k]
    return out

shared = {}
for subtype, excl_h, excl_m in [("Stro-7_C7 Fibroblast", (), ()),
                                ("Stro-8_dS1", (), ("decidual_apoptotic",)),
                                ("Stro-10_dS3", ("h3",), ()),
                                ("Stro-11_Stromal progenitor", (), ())]:
    hu = route_union("human", subtype, excl_h)
    mo = route_union("mouse", subtype, excl_m)
    shared[subtype] = sorted(hu & mo) if (hu and mo) else None
    routes_h = [k.split(":")[1].split("->")[0] for k in DRIVERS if k.startswith("human") and k.endswith(subtype)]
    routes_m = [k.split(":")[1].split("->")[0] for k in DRIVERS if k.startswith("mouse") and k.endswith(subtype)]
    print(f"{SHORT.get(subtype, subtype):22s} human routes {routes_h} x mouse routes {routes_m}"
          f" -> shared: {shared[subtype] if shared[subtype] is not None else '(no route in one species)'}")

verify("Stro-7 three-way conserved drivers", len(shared["Stro-7_C7 Fibroblast"]), 6, tol=0)
verify("Stro-8 three-way conserved drivers (decAp excluded)", len(shared["Stro-8_dS1"]), 2, tol=0)
verify("Stro-10 three-way conserved drivers (h3 excluded)", len(shared["Stro-10_dS3"]), 11, tol=0)
verify("Stro-11 has no human route, so no three-way claim", int(shared["Stro-11_Stromal progenitor"] is None), 1, tol=0)
verify("all Stro-7 shared drivers are identity-class",
       int(all(driver_class(g) == "identity" for g in shared["Stro-7_C7 Fibroblast"])), 1, tol=0)

#### 4c. Two matches are driven by the wrong kind of gene — flagged, because it changes the claim

- **decidual_apoptotic → Stro-8 (cosine 0.019): 7 of 15 top drivers are housekeeping** (RPS19,
  SEC61G, SERP1, UQCR11, ROMO1, EIF4G2, TMSB4X). Combined with §3.1's below-ceiling result, this is
  a **housekeeping artifact, not a correspondence**, and is reported as a non-match.
- **decidual_proliferative → Stro-11 (0.148): 6 of 15 drivers are mitotic** (ZWINT, GINS2, TFDP1,
  NCAPH2, DIAPH3, G2E3) — and **efib_proliferative → Stro-11 (0.347): 15 of 15 mitotic.** Both
  Stro-11 routes are **cell-cycle state matches, not identity matches**. This project has already
  established that the mouse "proliferative" decidual population is barely cycling at the median
  (the name is historical); a Stro-11 correspondence carried entirely by mitotic genes says the two
  populations share a cycling state, nothing more.
- By contrast **efib_outer → Stro-7 (0.243): 15 of 15 identity genes** (PTGIS, FAP, CCN5, EPHX1,
  FRZB, …) — the credible three-way anchor.

In [ ]:
d_ix = DRV.set_index(["query", "atlas_subtype"])
verify("decidual_apoptotic housekeeping drivers (of 15)", int(d_ix.loc[("decidual_apoptotic", "Stro-8 dS1"), "n_housekeeping"]), 7, tol=0)
verify("decidual_proliferative mitotic drivers (of 15)", int(d_ix.loc[("decidual_proliferative", "Stro-11 progenitor"), "n_cell_cycle"]), 6, tol=0)
verify("efib_proliferative mitotic drivers (of 15)", int(d_ix.loc[("efib_proliferative", "Stro-11 progenitor"), "n_cell_cycle"]), 15, tol=0)
verify("efib_outer identity drivers (of 15)", int(d_ix.loc[("efib_outer", "Stro-7 C7"), "n_identity"]), 15, tol=0)

## 5. CROSS-ARM SYNTHESIS — one table, figure-ready

Which Stro subtypes are recovered in both species, which in one, which in neither — with the driver
class carried into the verdict, per §4c. Written to `hammoud/TABLE_crossarm_synthesis.csv`.

In [ ]:
syn = pd.DataFrame([
    dict(subtype="Stro-5 RUNX3+", human_cluster=str(TBL.loc["Stro-5 RUNX3+", "best_query"]),
         human_cosine=float(TBL.loc["Stro-5 RUNX3+", "score"]), human_call=TBL.loc["Stro-5 RUNX3+", "call"],
         mouse_query="-", mouse_full_panel=np.nan, mouse_panel_cosine=np.nan, n_shared_drivers=0,
         shared_drivers="-", verdict="not recovered in either species (immune program absent in human "
                                     "by construction; mouse best is efib_middle_induced, UNRESOLVED)"),
    dict(subtype="Stro-7 C7", human_cluster=str(TBL.loc["Stro-7 C7", "best_query"]),
         human_cosine=float(TBL.loc["Stro-7 C7", "score"]), human_call=TBL.loc["Stro-7 C7", "call"],
         mouse_query="efib_outer", mouse_full_panel=float(prim.loc["eFib1", "score_full_panel"]),
         mouse_panel_cosine=float(prim.loc["eFib1", "json_panel_cosine"]),
         n_shared_drivers=len(shared["Stro-7_C7 Fibroblast"]),
         shared_drivers=", ".join(shared["Stro-7_C7 Fibroblast"]),
         verdict="recovered in both species, identity-driven - the three-way anchor"),
    dict(subtype="Stro-8 dS1", human_cluster=str(TBL.loc["Stro-8 dS1", "best_query"]),
         human_cosine=float(TBL.loc["Stro-8 dS1", "score"]), human_call=TBL.loc["Stro-8 dS1", "call"],
         mouse_query="decidual_mature", mouse_full_panel=float(prim.loc["dec3", "score_full_panel"]),
         mouse_panel_cosine=float(prim.loc["dec3", "json_panel_cosine"]),
         n_shared_drivers=len(shared["Stro-8_dS1"]), shared_drivers=", ".join(shared["Stro-8_dS1"]),
         verdict="human probable; mouse support panel-dependent (negative on the full panel)"),
    dict(subtype="Stro-9 dS2", human_cluster=str(TBL.loc["Stro-9 dS2", "best_query"]),
         human_cosine=float(TBL.loc["Stro-9 dS2", "score"]), human_call=TBL.loc["Stro-9 dS2", "call"],
         mouse_query="-", mouse_full_panel=np.nan, mouse_panel_cosine=np.nan, n_shared_drivers=0,
         shared_drivers="-", verdict="not resolved from dS1 in human; not recovered in mouse "
                                     "(held-out agreement 0.13)"),
    dict(subtype="Stro-10 dS3", human_cluster=str(TBL.loc["Stro-10 dS3", "best_query"]),
         human_cosine=float(TBL.loc["Stro-10 dS3", "score"]), human_call=TBL.loc["Stro-10 dS3", "call"],
         mouse_query="efib_subluminal (+efib_middle; decidual_boundary panel-dep.)",
         mouse_full_panel=float(prim.loc["eFib3", "score_full_panel"]),
         mouse_panel_cosine=float(prim.loc["eFib3", "json_panel_cosine"]),
         n_shared_drivers=len(shared["Stro-10_dS3"]), shared_drivers=", ".join(shared["Stro-10_dS3"]),
         verdict="recovered in both species; mouse signal diffuse across clusters"),
    dict(subtype="Stro-11 progenitor", human_cluster=str(TBL.loc["Stro-11 progenitor", "best_query"]),
         human_cosine=float(TBL.loc["Stro-11 progenitor", "score"]), human_call=TBL.loc["Stro-11 progenitor", "call"],
         mouse_query="efib_proliferative", mouse_full_panel=float(prim.loc["eFibP", "score_full_panel"]),
         mouse_panel_cosine=float(prim.loc["eFibP", "json_panel_cosine"]), n_shared_drivers=0,
         shared_drivers="- (no human route)",
         verdict="mouse only, and cell-cycle-driven: a state match, not an identity match"),
]).set_index("subtype")
display(syn.round(3))
syn.to_csv(HAM / "TABLE_crossarm_synthesis.csv")
back = pd.read_csv(HAM / "TABLE_crossarm_synthesis.csv", index_col=0)
verify("synthesis table written and read back (6 subtypes)", len(back), 6, tol=0)
verify("subtypes recovered in both species", int(syn["verdict"].str.startswith("recovered in both").sum()), 2, tol=0)
verify("Stro-11 verdict carries the state-match flag", int("state match" in syn.loc["Stro-11 progenitor", "verdict"]), 1, tol=0)

## 6. What to write in the rebuttal

**(i) Supported by both arms.**
- **Stro-7 C7 fibroblast.** Human: cluster 3, confident (0.343, reciprocal at every clustering
  resolution, 6/6 sections, bootstrap and held-out 1.00). Mouse: efib_outer (0.174 full panel /
  0.243 discriminative panel, above its negative-control ceiling, bootstrap 1.00, held-out 0.998).
  Six drivers are three-way conserved and all are identity genes: APOD, ITGA11, PTGIS, RERG, SCN7A,
  TNXB. This is the strongest claim in this body of work.
- **Stro-10 dS3.** Human: cluster 1, confident (0.389, 6/6 sections, 1.00/1.00). Mouse: recovered
  but diffuse (efib_subluminal and efib_middle on the full panel; decidual_boundary only on the
  discriminative panel). Eleven three-way conserved drivers, including WNT2, CDH11, NR2F2.

**(ii) Human only.** **Stro-8 dS1** — cluster 5, probable. The mouse counterpart (decidual_mature,
0.255 on the discriminative panel, 15/15 identity drivers, 2 shared with human) is real but
**panel-dependent**: see the caveat below.

**(iii) Mouse only.** **Stro-11 stromal progenitor** — efib_proliferative, reciprocal, 0.259/0.347.
**State the driver class**: all 15 top drivers are mitotic, so this is evidence of a shared cycling
state, not of a conserved progenitor identity. In human it has no counterpart because cycling nuclei
are absent by construction (§2.8).

**(iv) Not supported.** Stro-5 RUNX3+ (immune-defined; absent by construction in human, UNRESOLVED
in mouse). Stro-9 dS2 (not resolved from dS1 in human — an immediate-early/stress signature damped
in nuclei; not recovered in mouse). **decidual_apoptotic → Stro-8 is a housekeeping artifact** and
must not be cited as a correspondence. **h3 → Stro-10 is a negative-cosine argmax** and likewise.

**Caveat to state before a reviewer finds it — decidual matches are panel-dependent; fibroblast
matches are not.** On the full 15,606-gene ortholog panel the mouse decidual scores collapse
(decidual_mature is *negative*, decidual_proliferative ≈ 0) while every efib match keeps its sign
and rank on both panels. The decidual correspondences therefore exist only on the discriminative
panel — worth claiming, but as panel-conditional evidence, stated as such.

**Two-donor confound**, stated verbatim as in §2.5: donor and cycle phase are perfectly confounded;
six-section replication is replication across both donors and both phases, but cannot separate the
two factors.

**Prose-drift corrections made by this compilation** (do not quote the old numbers): the C7
enrichment block (§2.7 drift table — 14.4×, not 16.2×; 61.0%, not 58.2%; 9.8%, not 7.9%; 5.60%, not
6.63%) and the resolution claim (§2.1 — Stro-7 is reciprocal at all five resolutions but carries the
largest margin at only three of five).

## 7. Limits — stated, not softened

- **Single-cell atlas vs single-nucleus query**: compositional and dropout differences; this is why
  the contrastive cosine is the headline and why raw Spearman disagrees with it (§1).
- **n = 2 human donors**, perfectly confounded with cycle phase (§2.5) — this is literally one of
  the reviewer comments, and six-section replication answers it only partially.
- **Three Stro subtypes are undetectable in our human data by construction, not by biology**
  (§2.8): immune-defined Stro-5, cycling-defined Stro-11, stress-defined Stro-9.
- **Cosine on centroids is a similarity, not a statistical test.** The sink test (§2.4, §3.2), the
  gene bootstrap and the held-out splits are the evidence that a match is not an artifact; a
  double-centred cosine of 0.02 is not evidence of anything (§4a's excluded rows).
- **Any proliferative match driven by cell-cycle genes is a state match, not an identity match**
  (§4c) — this applies to both Stro-11 routes in the mouse.
- **The mouse arm's replication, marker-ablation, detectability and spatial analyses were never
  run** (§3.4). The mouse claims above rest on the bootstrap, the held-out splits, and the driver
  decomposition only.
- **The human driver decomposition (h0–h9) is from an unidentified 10-cluster vintage** (§4);
  cross-species driver claims are keyed on Stro subtype only.

## 8. Self-check summary — target: 0 MISMATCH

In [ ]:
summary = check_summary()
display(summary)
bad = summary[summary["status"].str.strip() == "MISMATCH"]
if len(bad):
    print("\nMISMATCHES (a failing verify is a result, not a bug to hide):")
    display(bad)
else:
    print("\nAll recorded claims reproduce from the stored artifacts.")